# One report through Gentrex

Gentrex turns the PDF of a genetic test report into one record, with four
commands. This notebook follows one benchmark report through them.

| Command | Reads | Writes |
| --- | --- | --- |
| `gentrex transcribe` | The PDF | The text of every page, with OCR for scanned pages |
| `gentrex select` | The page texts | Which pages to keep |
| `gentrex extract` | The kept pages | One record, with the values as printed |
| `gentrex normalize` | The record | Standard values: ISO dates, categories and IDs |

Make a run first ([`evaluation/`](../evaluation/README.md)), then open the
notebook from the Gentrex folder:

```bash
uv run --group notebook jupyter lab evaluation_results/walkthrough.ipynb
```

[analysis.ipynb](analysis.ipynb) has the results over all reports. The
tables here quote values from the report, so clear the outputs before you
share the notebook after a run on real reports.

In [ ]:
# The notebook group provides pandas and matplotlib.
# pyright: reportMissingImports=false

import benchruns as bench

runs = bench.load_runs()
bench.runs_table(runs)

`RUN` is the run to follow and `REPORT` one of its reports. By default they
are the newest run on the native PDFs, and its first report of which
`gentrex select` dropped a page. Put another folder name from the table or
another report name, as `"SYN-0004"`, in their place. A run on the scan
copy shows what OCR wrote.

In [ ]:
native = [name for name, run in runs.items() if bench.setup_of(run) == "native"]
RUN = (native or list(runs))[-1]
run = runs[RUN]
REPORT = bench.example_report(run)

files = bench.report_files(run, REPORT)
errors = bench.errors(RUN, REPORT)
bench.case_table(run, REPORT)

## The report

A benchmark report is synthetic. Code picked every value that it prints, so
its true content is known. The table above gives its template, kind of
test, result and challenges.

A blue frame marks the pages that `gentrex select` kept, and a dropped page
is faded.

In [ ]:
bench.pages_figure(files)

## `gentrex transcribe`

A page with a text layer is `native`, and its text is read from the PDF. A
page without one is `scanned` and goes to the OCR model. Text that is lost
here is lost for good.

In [ ]:
bench.pages_table(files)

## `gentrex select`

The model answers yes-or-no questions about every page: whether it states
the result, describes the patient's clinical features, or lists the dates
of the report. Each number is the probability of "yes". A page is kept when
one of them is at least 0.5.

In [ ]:
bench.decisions_table(files)

## `gentrex extract` and `gentrex normalize`

One row per field of the report. `In the report` is the truth,
`gentrex extract` the value as the model copied it, and `gentrex normalize`
the standard value. A shaded row has a value that isn't right, and
`Lost by` names the command that the scorer charges.

In [ ]:
bench.record_table(files, errors)

The same values on the pages that `gentrex extract` read. A blue box marks a
value that is right and a red one a value that isn't. A page of a scanned
copy has no text layer to search, so it has no boxes.

In [ ]:
bench.values_figure(files, errors)

The variants, as the report states them and as Gentrex extracted them, with
the IDs that `gentrex normalize` added. Fields ending in "as printed" hold
the report's own wording, and the field after each holds the closest of a
fixed list of values.

In [ ]:
bench.variants_table(files)

The patient's clinical features, each with its closest HPO term.

In [ ]:
bench.features_table(files)

## The score

The report counts as correct when nothing clinically relevant is wrong:
the result, the karyotype, and every variant with its gene, DNA change,
copy number, classification and zygosity. The table lists every value of
the report that isn't right.

In [ ]:
summary = run["by_report"][REPORT]
print(
    f"{REPORT} counts as {'correct' if summary['correct'] else 'wrong'}."
    f" {summary['errors']} of its values aren't right."
)
bench.errors_table(errors)